[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lshofsl/Advanced-topics-in-PoD-project-/blob/main/ER_NCA.ipynb)

In [ ]:
# Baseline NCA adapted
# @misc{guichard2025engramncaneuralcellularautomaton,
     # title={EngramNCA: a Neural Cellular Automaton Model of Memory Transfer},
     # author={Etienne Guichard and Felix Reimers and Mia Kvalsund and Mikkel Lepperød and Stefano Nichele},
     # year={2025},
     # eprint={2504.11855},
     # archivePrefix={arXiv},
     # primaryClass={cs.NE},
     # url={https://arxiv.org/abs/2504.11855},
#}



import os
import sys

!rm -rf Advanced-topics-in-PoD-project-
!git clone https://github.com/lshofsl/Advanced-topics-in-PoD-project-.git

repo_path = os.path.abspath("Advanced-topics-in-PoD-project-")
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

if os.path.exists(repo_path):
    print("Success! Files now found:", os.listdir(repo_path))



In [ ]:
#@title Imports { vertical-output: true}
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

from NCA import *
import utils as utils
from IPython.display import Image, HTML, clear_output
import torch.nn.functional as F
import logging
from IPython.display import display, HTML, Video
from PIL import Image
import cv2
import pandas as pd
from base64 import b64encode
from sklearn.decomposition import PCA
logger = logging.getLogger()
old_level = logger.level
logger.setLevel(100)

In [ ]:

#@title Setup { vertical-output: true}
HEIGHT = 35 #@param {type:"integer"}
WIDTH = 35 #@param {type:"integer"}
CHANNELS = 16 # @param {type:"integer"}<--- NCA feature channels
BATCH_SIZE = 12 #@param {type:"integer"}
PADDING = 5 #@param {type:"integer"}
POOL_SIZE = 2666 #@param {type:"integer"}<--- NCA training pool size, lower values train faster but are less stable
TRAINING_ITERS = 8000  #@param {type:"integer"}<-- Number of trainign iterations
HIDDEN_SIZE = 64 #@param {type:"integer"}<--- NCA hidden size


style = """
<style>
.output_wrapper, .output {
    display: flex;
    flex-direction: row-reverse; /* Align content to the right */
}
</style>
"""



In [ ]:
LIZARD = "Images/lizard.png"
path = LIZARD
image, image_to_display = utils.get_image(os.path.join(repo_path, path), HEIGHT, WIDTH, padding=PADDING)

HEIGHT = HEIGHT + 2*PADDING
WIDTH = WIDTH + 2*PADDING

In [ ]:
class SamplePool:
  def __init__(self, *, _parent=None, _parent_idx=None, **slots):
    self._parent = _parent
    self._parent_idx = _parent_idx
    self._slot_names = slots.keys()
    self._size = None
    for k, v in slots.items():
      if self._size is None:
        self._size = len(v)
      assert self._size == len(v)
      setattr(self, k, np.asarray(v))

  def sample(self, n):
    idx = np.random.choice(self._size, n, False)
    batch = {k: getattr(self, k)[idx] for k in self._slot_names}
    batch = SamplePool(**batch, _parent=self, _parent_idx=idx)
    return batch

  def commit(self):
    for k in self._slot_names:
      getattr(self._parent, k)[self._parent_idx] = getattr(self, k)



In [ ]:


#@title Display Primitives { vertical-output: true}
def make_seed(size, channels=16):
    x = np.zeros([channels, size, size], dtype=np.float32)
    x[3:, size // 2, size // 2] = 1.0
    return x

seed = make_seed(HEIGHT, CHANNELS)

pool_inputs = np.repeat(seed[None, ...], POOL_SIZE, axis=0)
pool = SamplePool(x=pool_inputs)


plt.imshow(image_to_display)
plt.show()



In [ ]:
batch = pool.sample(BATCH_SIZE)

In [ ]:
sobel_x = torch.tensor([[-1.0, 0.0, 1.0], [-2.0, 0.0, 2.0], [-1.0, 0.0, 1.0]], dtype=torch.float32, device=DEVICE)
lap = torch.tensor([[1.0, 2.0, 1.0], [2.0, -12, 2.0], [1.0, 2.0, 1.0]], dtype=torch.float32, device=DEVICE)
filters = torch.stack([sobel_x, sobel_x.T, lap])
folder = "weights"

In [ ]:
#@title Create Path for Saving Models { vertical-output: true}
path = "Trained_models/" + folder
if not os.path.exists(path):
    os.makedirs(path)
    print(f"Path: {path} created")
else:
    print(f"Path: {path} already exists, all OK!")


In [ ]:


#@title Initialise NCA { vertical-output: true}
base = image.tile(BATCH_SIZE, 1, 1, 1).to(DEVICE)
loss_log = []
nca = NCA_EBM(CHANNELS,HIDDEN_SIZE)
nca = nca.to(DEVICE)
optim = torch.optim.AdamW(nca.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=2000, gamma=0.3)
name = folder + "/" +type(nca).__name__



In [ ]:
def create_circular_mask(HEIGHT, WIDTH, center_x, center_y, radius, device="cuda:0"):
    height, width = HEIGHT, WIDTH
    y_grid, x_grid = torch.meshgrid(
        torch.arange(height, device=device),
        torch.arange(width, device=device),
        indexing='ij'
    )
    distance_squared = (x_grid - center_x) ** 2 + (y_grid - center_y) ** 2
    return distance_squared <= (radius ** 2)



In [ ]:
#@title Training { vertical-output: true}
DAMAGE_N = 0

lambda_mono = 0.1
lambda_drift = 0.5
K = 20


eta_log = []
j_norm_log = []
energy_loss = []
loss_mono_log = []
loss_drift_log = []

for i in range(TRAINING_ITERS + 1):
    # 1. Pool Sampling & Manipulation
    with torch.no_grad():
      batch = pool.sample(BATCH_SIZE)
      x0_np = batch.x.copy()  # Shape: (BATCH_SIZE, C, H, W)

      x0_tensor = torch.from_numpy(x0_np).to(DEVICE)
      pre_pixel_loss = (x0_tensor[:, :4, :, :] - base).pow(2).mean(dim=[1, 2, 3])

      # Sort indices (descending: highest loss first)
      loss_rank = pre_pixel_loss.cpu().numpy().argsort()[::-1]

      # Re-order x0_np AND align pool slot indices so write-back stays consistent
      x0_np = x0_np[loss_rank]

      # B. Replace highest-loss sample (index 0) with fresh seed
      seed_np = seed.detach().cpu().numpy() if isinstance(seed, torch.Tensor) else seed.copy()
      if seed_np.ndim == 4:
          seed_np = seed_np[0]

      x0_np[0] = seed_np

      # C. Apply damage to the BEST performing samples (lowest loss at the end of sorted array)
      if DAMAGE_N > 0:
          for idx in range(1, DAMAGE_N + 1):
              cx = random.randint(int(WIDTH * 0.2), int(WIDTH * 0.8))
              cy = random.randint(int(HEIGHT * 0.2), int(HEIGHT * 0.8))
              r = random.randint(4, 10)

              mask_np = create_circular_mask(HEIGHT, WIDTH, cx, cy, radius=r).cpu().numpy().astype(np.float32)

                # Ensure mask matches (1, H, W) or (C, H, W) explicitly
              if mask_np.ndim == 2:
                  mask_np = mask_np[None, ...]  # Shape: (1, H, W)

              damage = 1.0 - mask_np
              x0_np[-idx] *= damage

        # Send fully prepared initial state to GPU
      x = torch.from_numpy(x0_np).to(DEVICE)

    # 2. Rollout loop
    T = random.randrange(64, 94)
    energies = []
    for t in range(T):
        E_t = nca.energy(x)
        energies.append(E_t)
        x = nca(x)

    energies.append(nca.energy(x))

    pixel_loss = (x[:, :4, :, :] - base).pow(2).mean(dim=[1, 2, 3])
    edge_loss = (perchannel_conv(base, filters) - perchannel_conv(x[:, :4, :, :], filters)).pow(2).mean(dim=[1, 2, 3])

    per_sample_loss = pixel_loss + 0.5 * edge_loss
    loss_recon = per_sample_loss.mean()

    energies_t = torch.stack(energies[:-1])
    energies_t1 = torch.stack(energies[1:])
    loss_mono = torch.relu(energies_t1 - energies_t).mean()

    x_at_T = x.clone()
    loss_drift = 0.0
    for t in range(K):
        x = nca(x)


        # Calculate pixel and edge drift at step T + t
        drift_pixel = (x[:, :4, :, :] - base).pow(2).mean(dim=[1, 2, 3])
        drift_edge = (perchannel_conv(base, filters) - perchannel_conv(x[:, :4, :, :], filters)).pow(2).mean(dim=[1, 2, 3])

        # Combine per-sample drift and accumulate
        step_drift = drift_pixel + 0.5 * drift_edge
        loss_drift += step_drift.mean()

    # Normalize loss_drift over the K additional steps
    loss_drift = loss_drift / K


    loss = loss_recon + lambda_mono * loss_mono + lambda_drift * loss_drift
    loss.backward()
    torch.nn.utils.clip_grad_norm_(nca.parameters(), max_norm=1.0)
    optim.step()
    optim.zero_grad()

    eta_log.append(nca.eta.detach().item())
    j_norm_log.append(nca.J.norm().item())
    loss_mono_log.append(loss_mono.item())
    loss_drift_log.append(loss_drift.item())
    energy_loss.append(loss.item())
    loss_log.append(loss.log().item())

    with torch.no_grad():
      batch.x[:] = x_at_T.detach().cpu().numpy()
      batch.commit()

    scheduler.step()



    if i % 100 == 0:
        print(f"Training itter {i}, loss = {loss.item()}")
        plt.clf()
        clear_output()
        plt.figure(1, figsize=(10, 4))
        plt.title('Loss history')
        plt.plot(loss_log, '.', alpha=0.5, color="b")
        print("Batch")
        utils.show_batch(x_at_T[2:14])
        display(HTML(style))
        plt.show(block=False)
        plt.pause(0.01)

        torch.save(nca.state_dict(), "Trained_models/" + name + ".pth")
        print("Trained_models/" + name + ".pth")